# **meta-llama/Meta-Llama-3-8B-Instruct**

# *Llama APV0*

In [ ]:
#Connect with google drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
#install dependencies
!pip install -q -U transformers accelerate bitsandbytes tqdm
!pip install -q pandas==2.2.2
!pip install -q codecarbon==3.0.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 576.3/576.3 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.5/92.5 kB 11.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.63.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.


In [ ]:
#imports
import os
import ast
import time
import csv
from datetime import datetime
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
from codecarbon import EmissionsTracker

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

CODE_DIR = "/content/drive/MyDrive/HumanEval_Code_Test_Dataset"

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
PROMPT_NAME = "APV0"
MODEL_TAG = "Llama"

RUNS = 3
BATCH_SIZE = 5

WARMUP_SECONDS_CPU = 5 * 60
COOLING_SECONDS = 60

MODEL_ROOT_DIR = f"/content/drive/MyDrive/{MODEL_TAG}/{PROMPT_NAME}"
os.makedirs(MODEL_ROOT_DIR, exist_ok=True)

In [ ]:
#HuggingFace token access
login(token="hf_xxxxxxxxxxxxxxxxxxxxxxxxx")

In [ ]:
file_indices = range(164)
code_files = [os.path.join(CODE_DIR, f"HumanEval_{i}_code.py") for i in file_indices]
print(f"Found {len(code_files)} code files to process.")

num_batches = (len(code_files) + BATCH_SIZE - 1) // BATCH_SIZE
print("num_batches:", num_batches)

Found 164 code files to process.
num_batches: 33


In [ ]:
def extract_function_name(code_text):
    try:
        tree = ast.parse(code_text)
        function_names = [node.name for node in tree.body if isinstance(node, ast.FunctionDef)]
        return function_names[-1] if function_names else "unknown_function"
    except Exception:
        return "unknown_function"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

def warmup_cpu(duration_secs):
    print(f"\nCPU warm-up for {duration_secs} seconds...")
    start = time.time()
    a, b = 0, 1
    while (time.time() - start) < duration_secs:
        a, b = b, a + b
        if a > 10**7:
            a, b = 0, 1
    print("CPU warm-up finished.\n")

@torch.no_grad()
def warmup_model(tokenizer, model):
    print("Model/GPU warm-up (excluded): 2 tiny generations...")
    messages = [{"role": "user", "content": "Say hello in one word."}]
    inp = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
    _ = model.generate(input_ids=inp.input_ids, attention_mask=inp.attention_mask, max_new_tokens=8, do_sample=False)
    _ = model.generate(input_ids=inp.input_ids, attention_mask=inp.attention_mask, max_new_tokens=8, do_sample=False)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    print("Model/GPU warm-up finished.\n")

def backup_if_exists(path):
    if os.path.exists(path):
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        new_name = f"{path}.old_{ts}"
        os.rename(path, new_name)
        print(f"Existing file backed up: {new_name}")

In [ ]:
run_rows = []

for run_id in range(1, RUNS + 1):
    print(f"\n================ RUN {run_id}/{RUNS} ================\n")

    # ==========================================================
    # FIX #1: Per-run folders (no overlap)
    # ==========================================================
    RUN_DIR = os.path.join(MODEL_ROOT_DIR, f"run_{run_id}")
    TESTS_OUTPUT_DIR = os.path.join(RUN_DIR, "tests")
    SETUP_COST_DIR   = os.path.join(RUN_DIR, "setup_cost_csv")
    EMISSIONS_DIR    = os.path.join(RUN_DIR, "emissions_csv")
    RUNTIME_DIR      = os.path.join(RUN_DIR, "runtime_csv")
    SUMMARY_DIR      = os.path.join(RUN_DIR, "summary")

    os.makedirs(TESTS_OUTPUT_DIR, exist_ok=True)
    os.makedirs(SETUP_COST_DIR, exist_ok=True)
    os.makedirs(EMISSIONS_DIR, exist_ok=True)
    os.makedirs(RUNTIME_DIR, exist_ok=True)
    os.makedirs(SUMMARY_DIR, exist_ok=True)

    setup_cost_csv_path = os.path.join(SETUP_COST_DIR, f"setup_cost_run_{run_id}.csv")
    emissions_csv_path  = os.path.join(EMISSIONS_DIR,  f"emissions_run_{run_id}.csv")
    runtime_csv_path    = os.path.join(RUNTIME_DIR,    f"runtime_run_{run_id}.csv")

    # Ensure no overwrite/mixing if you re-run the notebook
    backup_if_exists(setup_cost_csv_path)
    backup_if_exists(emissions_csv_path)
    backup_if_exists(runtime_csv_path)

    run_start_time = time.time()

    # (Excluded) CPU warm-up
    warmup_cpu(WARMUP_SECONDS_CPU)

    # ----------------------------------------------------------
    # Setup cost tracking: model loading only
    # ----------------------------------------------------------
    setup_tracker = EmissionsTracker(
        project_name=f"{MODEL_ID.replace('/', '_')}_run_{run_id}_setup_loading",
        output_dir=os.path.dirname(setup_cost_csv_path),
        output_file=os.path.basename(setup_cost_csv_path),
    )
    setup_tracker.start()

    print(f"Loading 8-bit quantized model '{MODEL_ID}'...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto"
    )
    model.eval()

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    setup_emissions_kg = setup_tracker.stop()
    print(f"Run {run_id} setup emissions (kg CO2eq): {setup_emissions_kg}")
    print(f"Run {run_id} setup cost CSV: {setup_cost_csv_path}")

    # (Excluded) Model/GPU warm-up after load
    warmup_model(tokenizer, model)

    # runtime CSV header
    with open(runtime_csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["run_id", "batch_id", "file_path", "status", "seconds"])

        # ==========================================================
        # FIX #2: Per-batch CodeCarbon tracking (one row per batch)
        # This appends incrementally to emissions_run_{run_id}.csv
        # ==========================================================
        for batch_id in tqdm(range(num_batches), desc=f"Run {run_id} Processing batches"):
            batch_start = time.time()

            batch_tracker = EmissionsTracker(
                project_name=f"{MODEL_ID.replace('/', '_')}_run_{run_id}_batch_{batch_id}_inference",
                output_dir=os.path.dirname(emissions_csv_path),
                output_file=os.path.basename(emissions_csv_path),
            )
            batch_tracker.start()

            start_index = batch_id * BATCH_SIZE
            end_index = min(start_index + BATCH_SIZE, len(code_files))
            batch_files = code_files[start_index:end_index]

            for file_path in batch_files:
                t0 = time.time()

                if not os.path.exists(file_path):
                    w.writerow([run_id, batch_id, file_path, "missing", time.time() - t0])
                    continue

                try:
                    with open(file_path, "r", encoding="utf-8") as src:
                        code_content = src.read()

                    module_name = os.path.basename(file_path).replace(".py", "")
                    function_name = extract_function_name(code_content)

                    messages = [{
                        "role": "user",
                        "content": f"""Generate a unittest test script for the following Python function.
The script should fully test the function and be runnable directly.

### Output Formatting
1. Start with: import unittest
2. Include: from {module_name} import {function_name}
3. End with:
if __name__ == '__main__':
    unittest.main()

Function:
{code_content}
"""
                    }]

                    model_inputs = tokenizer.apply_chat_template(
                        messages,
                        add_generation_prompt=True,
                        return_tensors="pt"
                    ).to(model.device)

                    generated_ids = model.generate(
                        input_ids=model_inputs.input_ids,
                        attention_mask=model_inputs.attention_mask,
                        max_new_tokens=1024,
                        do_sample=False,
                        temperature=0.0
                    )

                    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
                    generated_test = generated_text.strip().replace("```python", "").replace("```", "").strip()

                    task_id = os.path.basename(file_path).replace("_code.py", "")
                    output_filename = f"test_{task_id}_test.py"
                    output_path = os.path.join(TESTS_OUTPUT_DIR, output_filename)

                    with open(output_path, "w", encoding="utf-8") as out:
                        out.write(generated_test)

                    w.writerow([run_id, batch_id, file_path, "ok", time.time() - t0])

                except Exception as e:
                    w.writerow([run_id, batch_id, file_path, f"error: {type(e).__name__}", time.time() - t0])

            # finish batch tracker (this writes ONE row into emissions_run_{run_id}.csv)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            _ = batch_tracker.stop()

            w.writerow([run_id, batch_id, f"[BATCH_TOTAL_{batch_id}]", "batch_done", time.time() - batch_start])
            f.flush()  # ensure runtime rows are saved even if session dies

    run_duration_s = time.time() - run_start_time

    run_rows.append({
        "run_id": run_id,
        "run_dir": RUN_DIR,
        "setup_cost_csv": setup_cost_csv_path,
        "emissions_csv": emissions_csv_path,
        "runtime_csv": runtime_csv_path,
        "run_duration_seconds": run_duration_s
    })

    print(f"\nRun {run_id} completed. Duration (s): {run_duration_s}")
    print(f"Run {run_id} emissions CSV (per-batch rows): {emissions_csv_path}")
    print(f"Run {run_id} runtime CSV: {runtime_csv_path}")

    if run_id < RUNS:
        print(f"\nCooling for {COOLING_SECONDS} seconds...\n")
        try:
            del model
            del tokenizer
        except Exception:
            pass
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        time.sleep(COOLING_SECONDS)

print("\nAll runs complete!")


================ RUN 1/3 ================


CPU warm-up for 300 seconds...


[codecarbon WARNING @ 03:38:20] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 03:38:20] [setup] RAM Tracking...
[codecarbon INFO @ 03:38:20] [setup] CPU Tracking...


CPU warm-up finished.



[codecarbon WARNING @ 03:38:22] We saw that you have a Intel(R) Xeon(R) CPU @ 2.20GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 03:38:22] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 03:38:22] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.20GHz
[codecarbon WARNING @ 03:38:22] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 03:38:22] [setup] GPU Tracking...
[codecarbon INFO @ 03:38:22] Tracking Nvidia GPU via pynvml
[codecarbon INFO @ 03:38:22] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Method: global constant
                GPU Tracking Method: pynvml
            
[codecarbon INFO @ 03:38:22] >>> Tracker's metadata:
[codecarbon INFO @ 03:38:22]   Platform sys

Loading 8-bit quantized model 'meta-llama/Meta-Llama-3-8B-Instruct'...


...

Run 3 setup emissions (kg CO2eq): 0.0002943078948688717
Run 3 setup cost CSV: /content/drive/MyDrive/Llama/APV0/run_3/setup_cost_csv/setup_cost_run_3.csv
Model/GPU warm-up (excluded): 2 tiny generations...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Model/GPU warm-up finished.



Streaming output truncated to the last 5000 lines.
[codecarbon INFO @ 11:04:17] Energy consumed for All CPU : 0.000885 kWh
[codecarbon INFO @ 11:04:17] Energy consumed for all GPUs : 0.001158 kWh. Total GPU Power : 60.301337083371955 W
[codecarbon INFO @ 11:04:17] 0.002250 kWh of electricity used since the beginning.
[codecarbon INFO @ 11:04:32] Energy consumed for RAM : 0.000250 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 11:04:32] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 11:04:32] Energy consumed for All CPU : 0.001062 kWh
[codecarbon INFO @ 11:04:32] Energy consumed for all GPUs : 0.001368 kWh. Total GPU Power : 50.391818817433986 W
[codecarbon INFO @ 11:04:32] 0.002679 kWh of electricity used since the beginning.
[codecarbon INFO @ 11:04:47] Energy consumed for RAM : 0.000291 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 11:04:47] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 11:04:47] 


Run 3 completed. Duration (s): 12755.60408449173
Run 3 emissions CSV (per-batch rows): /content/drive/MyDrive/Llama/APV0/run_3/emissions_csv/emissions_run_3.csv
Run 3 runtime CSV: /content/drive/MyDrive/Llama/APV0/run_3/runtime_csv/runtime_run_3.csv

All runs complete!


# *Phi APV1*

In [ ]:
#Connect with google drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
#install dependencies
!pip install -q -U transformers accelerate bitsandbytes tqdm
!pip install -q pandas==2.2.2
!pip install -q codecarbon==3.0.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 576.3/576.3 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.5/92.5 kB 12.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.63.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.


In [ ]:
#imports
import os
import ast
import time
import csv
from datetime import datetime
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
from codecarbon import EmissionsTracker

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

CODE_DIR = "/content/drive/MyDrive/HumanEval_Code_Test_Dataset"

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
PROMPT_NAME = "APV1"
MODEL_TAG = "Llama"

RUNS = 1
BATCH_SIZE = 5

WARMUP_SECONDS_CPU = 5 * 60
COOLING_SECONDS = 60

MODEL_ROOT_DIR = f"/content/drive/MyDrive/{MODEL_TAG}/{PROMPT_NAME}"
os.makedirs(MODEL_ROOT_DIR, exist_ok=True)

In [ ]:
#HuggingFace token access
login(token="hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")

In [ ]:
file_indices = range(164)
code_files = [os.path.join(CODE_DIR, f"HumanEval_{i}_code.py") for i in file_indices]
print(f"Found {len(code_files)} code files to process.")

num_batches = (len(code_files) + BATCH_SIZE - 1) // BATCH_SIZE
print("num_batches:", num_batches)

Found 164 code files to process.
num_batches: 33


In [ ]:
def extract_function_name(code_text):
    try:
        tree = ast.parse(code_text)
        function_names = [node.name for node in tree.body if isinstance(node, ast.FunctionDef)]
        return function_names[-1] if function_names else "unknown_function"
    except Exception:
        return "unknown_function"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

def warmup_cpu(duration_secs):
    print(f"\nCPU warm-up for {duration_secs} seconds...")
    start = time.time()
    a, b = 0, 1
    while (time.time() - start) < duration_secs:
        a, b = b, a + b
        if a > 10**7:
            a, b = 0, 1
    print("CPU warm-up finished.\n")

@torch.no_grad()
def warmup_model(tokenizer, model):
    print("Model/GPU warm-up (excluded): 2 tiny generations...")
    messages = [{"role": "user", "content": "Say hello in one word."}]
    inp = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
    _ = model.generate(input_ids=inp.input_ids, attention_mask=inp.attention_mask, max_new_tokens=8, do_sample=False)
    _ = model.generate(input_ids=inp.input_ids, attention_mask=inp.attention_mask, max_new_tokens=8, do_sample=False)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    print("Model/GPU warm-up finished.\n")

def backup_if_exists(path):
    if os.path.exists(path):
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        new_name = f"{path}.old_{ts}"
        os.rename(path, new_name)
        print(f"Existing file backed up: {new_name}")

In [ ]:
run_rows = []

for run_id in range(1, RUNS + 1):
    print(f"\n================ RUN {run_id}/{RUNS} ================\n")

    # ==========================================================
    # FIX #1: Per-run folders (no overlap)
    # ==========================================================
    RUN_DIR = os.path.join(MODEL_ROOT_DIR, f"run_{run_id}")
    TESTS_OUTPUT_DIR = os.path.join(RUN_DIR, "tests")
    SETUP_COST_DIR   = os.path.join(RUN_DIR, "setup_cost_csv")
    EMISSIONS_DIR    = os.path.join(RUN_DIR, "emissions_csv")
    RUNTIME_DIR      = os.path.join(RUN_DIR, "runtime_csv")
    SUMMARY_DIR      = os.path.join(RUN_DIR, "summary")

    os.makedirs(TESTS_OUTPUT_DIR, exist_ok=True)
    os.makedirs(SETUP_COST_DIR, exist_ok=True)
    os.makedirs(EMISSIONS_DIR, exist_ok=True)
    os.makedirs(RUNTIME_DIR, exist_ok=True)
    os.makedirs(SUMMARY_DIR, exist_ok=True)

    setup_cost_csv_path = os.path.join(SETUP_COST_DIR, f"setup_cost_run_{run_id}.csv")
    emissions_csv_path  = os.path.join(EMISSIONS_DIR,  f"emissions_run_{run_id}.csv")
    runtime_csv_path    = os.path.join(RUNTIME_DIR,    f"runtime_run_{run_id}.csv")

    # Ensure no overwrite/mixing if you re-run the notebook
    backup_if_exists(setup_cost_csv_path)
    backup_if_exists(emissions_csv_path)
    backup_if_exists(runtime_csv_path)

    run_start_time = time.time()

    # (Excluded) CPU warm-up
    warmup_cpu(WARMUP_SECONDS_CPU)

    # ----------------------------------------------------------
    # Setup cost tracking: model loading only
    # ----------------------------------------------------------
    setup_tracker = EmissionsTracker(
        project_name=f"{MODEL_ID.replace('/', '_')}_run_{run_id}_setup_loading",
        output_dir=os.path.dirname(setup_cost_csv_path),
        output_file=os.path.basename(setup_cost_csv_path),
    )
    setup_tracker.start()

    print(f"Loading 8-bit quantized model '{MODEL_ID}'...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto"
    )
    model.eval()

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    setup_emissions_kg = setup_tracker.stop()
    print(f"Run {run_id} setup emissions (kg CO2eq): {setup_emissions_kg}")
    print(f"Run {run_id} setup cost CSV: {setup_cost_csv_path}")

    # (Excluded) Model/GPU warm-up after load
    warmup_model(tokenizer, model)

    # runtime CSV header
    with open(runtime_csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["run_id", "batch_id", "file_path", "status", "seconds"])

        # ==========================================================
        # FIX #2: Per-batch CodeCarbon tracking (one row per batch)
        # This appends incrementally to emissions_run_{run_id}.csv
        # ==========================================================
        for batch_id in tqdm(range(num_batches), desc=f"Run {run_id} Processing batches"):
            batch_start = time.time()

            batch_tracker = EmissionsTracker(
                project_name=f"{MODEL_ID.replace('/', '_')}_run_{run_id}_batch_{batch_id}_inference",
                output_dir=os.path.dirname(emissions_csv_path),
                output_file=os.path.basename(emissions_csv_path),
            )
            batch_tracker.start()

            start_index = batch_id * BATCH_SIZE
            end_index = min(start_index + BATCH_SIZE, len(code_files))
            batch_files = code_files[start_index:end_index]

            for file_path in batch_files:
                t0 = time.time()

                if not os.path.exists(file_path):
                    w.writerow([run_id, batch_id, file_path, "missing", time.time() - t0])
                    continue

                try:
                    with open(file_path, "r", encoding="utf-8") as src:
                        code_content = src.read()

                    module_name = os.path.basename(file_path).replace(".py", "")
                    function_name = extract_function_name(code_content)

                    messages = [{
                        "role": "user",
                    "content": f"""You are an expert Python programmer.
Your task is to write a comprehensive unittest test suite for the given Python function.

### Output Formatting
1. Start with: import unittest
2. Include: from {module_name} import {function_name}

3. End with:
if __name__ == '__main__':
    unittest.main()

Function:
{code_content}
"""
                    }]

                    model_inputs = tokenizer.apply_chat_template(
                        messages,
                        add_generation_prompt=True,
                        return_tensors="pt"
                    ).to(model.device)

                    generated_ids = model.generate(
                        input_ids=model_inputs.input_ids,
                        attention_mask=model_inputs.attention_mask,
                        max_new_tokens=1024,
                        do_sample=False,
                        temperature=0.0
                    )

                    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
                    generated_test = generated_text.strip().replace("```python", "").replace("```", "").strip()

                    task_id = os.path.basename(file_path).replace("_code.py", "")
                    output_filename = f"test_{task_id}_test.py"
                    output_path = os.path.join(TESTS_OUTPUT_DIR, output_filename)

                    with open(output_path, "w", encoding="utf-8") as out:
                        out.write(generated_test)

                    w.writerow([run_id, batch_id, file_path, "ok", time.time() - t0])

                except Exception as e:
                    w.writerow([run_id, batch_id, file_path, f"error: {type(e).__name__}", time.time() - t0])

            # finish batch tracker (this writes ONE row into emissions_run_{run_id}.csv)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            _ = batch_tracker.stop()

            w.writerow([run_id, batch_id, f"[BATCH_TOTAL_{batch_id}]", "batch_done", time.time() - batch_start])
            f.flush()  # ensure runtime rows are saved even if session dies

    run_duration_s = time.time() - run_start_time

    run_rows.append({
        "run_id": run_id,
        "run_dir": RUN_DIR,
        "setup_cost_csv": setup_cost_csv_path,
        "emissions_csv": emissions_csv_path,
        "runtime_csv": runtime_csv_path,
        "run_duration_seconds": run_duration_s
    })

    print(f"\nRun {run_id} completed. Duration (s): {run_duration_s}")
    print(f"Run {run_id} emissions CSV (per-batch rows): {emissions_csv_path}")
    print(f"Run {run_id} runtime CSV: {runtime_csv_path}")

    if run_id < RUNS:
        print(f"\nCooling for {COOLING_SECONDS} seconds...\n")
        try:
            del model
            del tokenizer
        except Exception:
            pass
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        time.sleep(COOLING_SECONDS)

print("\nAll runs complete!")


================ RUN 1/1 ================


CPU warm-up for 300 seconds...


[codecarbon WARNING @ 15:24:21] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 15:24:21] [setup] RAM Tracking...
[codecarbon INFO @ 15:24:21] [setup] CPU Tracking...


CPU warm-up finished.



[codecarbon WARNING @ 15:24:22] We saw that you have a Intel(R) Xeon(R) CPU @ 2.00GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 15:24:22] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 15:24:22] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.00GHz
[codecarbon WARNING @ 15:24:22] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 15:24:22] [setup] GPU Tracking...
[codecarbon INFO @ 15:24:22] Tracking Nvidia GPU via pynvml
[codecarbon INFO @ 15:24:22] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Method: global constant
                GPU Tracking Method: pynvml
            
[codecarbon INFO @ 15:24:22] >>> Tracker's metadata:
[codecarbon INFO @ 15:24:22]   Platform sys

Loading 8-bit quantized model 'meta-llama/Meta-Llama-3-8B-Instruct'...


...

Run 1 setup emissions (kg CO2eq): 0.00196439860311804
Run 1 setup cost CSV: /content/drive/MyDrive/Llama/APV1/run_1/setup_cost_csv/setup_cost_run_1.csv
Model/GPU warm-up (excluded): 2 tiny generations...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Model/GPU warm-up finished.



Streaming output truncated to the last 5000 lines.
[codecarbon INFO @ 16:39:07] Energy consumed for RAM : 0.000416 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 16:39:07] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 16:39:07] Energy consumed for All CPU : 0.001769 kWh
[codecarbon INFO @ 16:39:07] Energy consumed for all GPUs : 0.002266 kWh. Total GPU Power : 51.93743991228947 W
[codecarbon INFO @ 16:39:07] 0.004452 kWh of electricity used since the beginning.
[codecarbon INFO @ 16:39:22] Energy consumed for RAM : 0.000458 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 16:39:22] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 16:39:22] Energy consumed for All CPU : 0.001946 kWh
[codecarbon INFO @ 16:39:22] Energy consumed for all GPUs : 0.002468 kWh. Total GPU Power : 48.404491721846455 W
[codecarbon INFO @ 16:39:22] 0.004872 kWh of electricity used since the beginning.
[codecarbon INFO @ 16:39:37] E


Run 1 completed. Duration (s): 16963.180641651154
Run 1 emissions CSV (per-batch rows): /content/drive/MyDrive/Llama/APV1/run_1/emissions_csv/emissions_run_1.csv
Run 1 runtime CSV: /content/drive/MyDrive/Llama/APV1/run_1/runtime_csv/runtime_run_1.csv

All runs complete!


# *Phi APV2*

In [ ]:
#Connect with google drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
#install dependencies
!pip install -q -U transformers accelerate bitsandbytes tqdm
!pip install -q pandas==2.2.2
!pip install -q codecarbon==3.0.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 576.3/576.3 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 575.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.5/92.5 kB 11.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.63.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.


In [ ]:
#imports
import os
import ast
import time
import csv
from datetime import datetime
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
from codecarbon import EmissionsTracker

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

CODE_DIR = "/content/drive/MyDrive/HumanEval_Code_Test_Dataset"

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
PROMPT_NAME = "APV2"
MODEL_TAG = "Llama"

RUNS = 1
BATCH_SIZE = 5

WARMUP_SECONDS_CPU = 5 * 60
COOLING_SECONDS = 60

MODEL_ROOT_DIR = f"/content/drive/MyDrive/{MODEL_TAG}/{PROMPT_NAME}"
os.makedirs(MODEL_ROOT_DIR, exist_ok=True)

In [ ]:
#HuggingFace token access
login(token="hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")

In [ ]:
file_indices = range(164)
code_files = [os.path.join(CODE_DIR, f"HumanEval_{i}_code.py") for i in file_indices]
print(f"Found {len(code_files)} code files to process.")

num_batches = (len(code_files) + BATCH_SIZE - 1) // BATCH_SIZE
print("num_batches:", num_batches)

Found 164 code files to process.
num_batches: 33


In [ ]:
def extract_function_name(code_text):
    try:
        tree = ast.parse(code_text)
        function_names = [node.name for node in tree.body if isinstance(node, ast.FunctionDef)]
        return function_names[-1] if function_names else "unknown_function"
    except Exception:
        return "unknown_function"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

def warmup_cpu(duration_secs):
    print(f"\nCPU warm-up for {duration_secs} seconds...")
    start = time.time()
    a, b = 0, 1
    while (time.time() - start) < duration_secs:
        a, b = b, a + b
        if a > 10**7:
            a, b = 0, 1
    print("CPU warm-up finished.\n")

@torch.no_grad()
def warmup_model(tokenizer, model):
    print("Model/GPU warm-up (excluded): 2 tiny generations...")
    messages = [{"role": "user", "content": "Say hello in one word."}]
    inp = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
    _ = model.generate(input_ids=inp.input_ids, attention_mask=inp.attention_mask, max_new_tokens=8, do_sample=False)
    _ = model.generate(input_ids=inp.input_ids, attention_mask=inp.attention_mask, max_new_tokens=8, do_sample=False)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    print("Model/GPU warm-up finished.\n")

def backup_if_exists(path):
    if os.path.exists(path):
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        new_name = f"{path}.old_{ts}"
        os.rename(path, new_name)
        print(f"Existing file backed up: {new_name}")

In [ ]:
run_rows = []

for run_id in range(1, RUNS + 1):
    print(f"\n================ RUN {run_id}/{RUNS} ================\n")

    # ==========================================================
    # FIX #1: Per-run folders (no overlap)
    # ==========================================================
    RUN_DIR = os.path.join(MODEL_ROOT_DIR, f"run_{run_id}")
    TESTS_OUTPUT_DIR = os.path.join(RUN_DIR, "tests")
    SETUP_COST_DIR   = os.path.join(RUN_DIR, "setup_cost_csv")
    EMISSIONS_DIR    = os.path.join(RUN_DIR, "emissions_csv")
    RUNTIME_DIR      = os.path.join(RUN_DIR, "runtime_csv")
    SUMMARY_DIR      = os.path.join(RUN_DIR, "summary")

    os.makedirs(TESTS_OUTPUT_DIR, exist_ok=True)
    os.makedirs(SETUP_COST_DIR, exist_ok=True)
    os.makedirs(EMISSIONS_DIR, exist_ok=True)
    os.makedirs(RUNTIME_DIR, exist_ok=True)
    os.makedirs(SUMMARY_DIR, exist_ok=True)

    setup_cost_csv_path = os.path.join(SETUP_COST_DIR, f"setup_cost_run_{run_id}.csv")
    emissions_csv_path  = os.path.join(EMISSIONS_DIR,  f"emissions_run_{run_id}.csv")
    runtime_csv_path    = os.path.join(RUNTIME_DIR,    f"runtime_run_{run_id}.csv")

    # Ensure no overwrite/mixing if you re-run the notebook
    backup_if_exists(setup_cost_csv_path)
    backup_if_exists(emissions_csv_path)
    backup_if_exists(runtime_csv_path)

    run_start_time = time.time()

    # (Excluded) CPU warm-up
    warmup_cpu(WARMUP_SECONDS_CPU)

    # ----------------------------------------------------------
    # Setup cost tracking: model loading only
    # ----------------------------------------------------------
    setup_tracker = EmissionsTracker(
        project_name=f"{MODEL_ID.replace('/', '_')}_run_{run_id}_setup_loading",
        output_dir=os.path.dirname(setup_cost_csv_path),
        output_file=os.path.basename(setup_cost_csv_path),
    )
    setup_tracker.start()

    print(f"Loading 8-bit quantized model '{MODEL_ID}'...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto"
    )
    model.eval()

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    setup_emissions_kg = setup_tracker.stop()
    print(f"Run {run_id} setup emissions (kg CO2eq): {setup_emissions_kg}")
    print(f"Run {run_id} setup cost CSV: {setup_cost_csv_path}")

    # (Excluded) Model/GPU warm-up after load
    warmup_model(tokenizer, model)

    # runtime CSV header
    with open(runtime_csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["run_id", "batch_id", "file_path", "status", "seconds"])

        # ==========================================================
        # FIX #2: Per-batch CodeCarbon tracking (one row per batch)
        # This appends incrementally to emissions_run_{run_id}.csv
        # ==========================================================
        for batch_id in tqdm(range(num_batches), desc=f"Run {run_id} Processing batches"):
            batch_start = time.time()

            batch_tracker = EmissionsTracker(
                project_name=f"{MODEL_ID.replace('/', '_')}_run_{run_id}_batch_{batch_id}_inference",
                output_dir=os.path.dirname(emissions_csv_path),
                output_file=os.path.basename(emissions_csv_path),
            )
            batch_tracker.start()

            start_index = batch_id * BATCH_SIZE
            end_index = min(start_index + BATCH_SIZE, len(code_files))
            batch_files = code_files[start_index:end_index]

            for file_path in batch_files:
                t0 = time.time()

                if not os.path.exists(file_path):
                    w.writerow([run_id, batch_id, file_path, "missing", time.time() - t0])
                    continue

                try:
                    with open(file_path, "r", encoding="utf-8") as src:
                        code_content = src.read()

                    module_name = os.path.basename(file_path).replace(".py", "")
                    function_name = extract_function_name(code_content)

                    messages = [
                {
                    "role": "system",
                    "content": "You are an expert Python programmer whose primary role is to write comprehensive unittest test suites. Be professional, precise, and output only runnable Python code."
                },
                {
                    "role": "user",
                    "content": f"""Write a complete unittest test suite for the following Python function.
Follow all rules carefully.

### Output Formatting
1. Start with: import unittest
2. Include: from {module_name} import {function_name}
3. End with:
if __name__ == '__main__':
    unittest.main()

Function:
{code_content}
"""
                    }]

                    model_inputs = tokenizer.apply_chat_template(
                        messages,
                        add_generation_prompt=True,
                        return_tensors="pt"
                    ).to(model.device)

                    generated_ids = model.generate(
                        input_ids=model_inputs.input_ids,
                        attention_mask=model_inputs.attention_mask,
                        max_new_tokens=1024,
                        do_sample=False,
                        temperature=0.0
                    )

                    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
                    generated_test = generated_text.strip().replace("```python", "").replace("```", "").strip()

                    task_id = os.path.basename(file_path).replace("_code.py", "")
                    output_filename = f"test_{task_id}_test.py"
                    output_path = os.path.join(TESTS_OUTPUT_DIR, output_filename)

                    with open(output_path, "w", encoding="utf-8") as out:
                        out.write(generated_test)

                    w.writerow([run_id, batch_id, file_path, "ok", time.time() - t0])

                except Exception as e:
                    w.writerow([run_id, batch_id, file_path, f"error: {type(e).__name__}", time.time() - t0])

            # finish batch tracker (this writes ONE row into emissions_run_{run_id}.csv)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            _ = batch_tracker.stop()

            w.writerow([run_id, batch_id, f"[BATCH_TOTAL_{batch_id}]", "batch_done", time.time() - batch_start])
            f.flush()  # ensure runtime rows are saved even if session dies

    run_duration_s = time.time() - run_start_time

    run_rows.append({
        "run_id": run_id,
        "run_dir": RUN_DIR,
        "setup_cost_csv": setup_cost_csv_path,
        "emissions_csv": emissions_csv_path,
        "runtime_csv": runtime_csv_path,
        "run_duration_seconds": run_duration_s
    })

    print(f"\nRun {run_id} completed. Duration (s): {run_duration_s}")
    print(f"Run {run_id} emissions CSV (per-batch rows): {emissions_csv_path}")
    print(f"Run {run_id} runtime CSV: {runtime_csv_path}")

    if run_id < RUNS:
        print(f"\nCooling for {COOLING_SECONDS} seconds...\n")
        try:
            del model
            del tokenizer
        except Exception:
            pass
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        time.sleep(COOLING_SECONDS)

print("\nAll runs complete!")


================ RUN 1/3 ================


CPU warm-up for 300 seconds...


[codecarbon WARNING @ 21:28:01] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 21:28:01] [setup] RAM Tracking...
[codecarbon INFO @ 21:28:01] [setup] CPU Tracking...


CPU warm-up finished.



[codecarbon WARNING @ 21:28:02] We saw that you have a Intel(R) Xeon(R) CPU @ 2.00GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 21:28:02] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 21:28:02] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.00GHz
[codecarbon WARNING @ 21:28:02] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 21:28:02] [setup] GPU Tracking...
[codecarbon INFO @ 21:28:02] Tracking Nvidia GPU via pynvml
[codecarbon INFO @ 21:28:02] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Method: global constant
                GPU Tracking Method: pynvml
            
[codecarbon INFO @ 21:28:02] >>> Tracker's metadata:
[codecarbon INFO @ 21:28:02]   Platform sys

Loading 8-bit quantized model 'meta-llama/Meta-Llama-3-8B-Instruct'...


...

Run 2 setup emissions (kg CO2eq): 0.0007266396735186295
Run 2 setup cost CSV: /content/drive/MyDrive/Llama/APV2/run_2/setup_cost_csv/setup_cost_run_2.csv
Model/GPU warm-up (excluded): 2 tiny generations...


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Model/GPU warm-up finished.



Run 2 Processing batches:   0%|          | 0/33 [00:00<?, ?it/s][codecarbon WARNING @ 00:23:45] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 00:23:45] [setup] RAM Tracking...
[codecarbon INFO @ 00:23:45] [setup] CPU Tracking...
[codecarbon WARNING @ 00:23:46] We saw that you have a Intel(R) Xeon(R) CPU @ 2.00GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 00:23:46] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 00:23:46] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.00GHz
[codecarbon WARNING @ 00:23:46] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 00:23:46] [setup] GPU Tracking...
[codecarbon INFO @ 00:23:46] Tracking Nvidia GPU via pynvml
[codecarbon INFO @ 00:23:46] The below tracking methods have been set up

KeyboardInterrupt: 

# *Phi APV3*

In [ ]:
#Connect with google drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
#install dependencies
!pip install -q -U transformers accelerate bitsandbytes tqdm
!pip install -q pandas==2.2.2
!pip install -q codecarbon==3.0.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 576.3/576.3 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.5/92.5 kB 12.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.63.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.


In [ ]:
#imports
import os
import ast
import time
import csv
from datetime import datetime
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
from codecarbon import EmissionsTracker

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

CODE_DIR = "/content/drive/MyDrive/HumanEval_Code_Test_Dataset"

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
PROMPT_NAME = "APV3"
MODEL_TAG = "Llama"

RUNS = 1
BATCH_SIZE = 5

WARMUP_SECONDS_CPU = 5 * 60
COOLING_SECONDS = 60

MODEL_ROOT_DIR = f"/content/drive/MyDrive/{MODEL_TAG}/{PROMPT_NAME}"
os.makedirs(MODEL_ROOT_DIR, exist_ok=True)

In [ ]:
#HuggingFace token access
login(token="hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")

In [ ]:
file_indices = range(164)
code_files = [os.path.join(CODE_DIR, f"HumanEval_{i}_code.py") for i in file_indices]
print(f"Found {len(code_files)} code files to process.")

num_batches = (len(code_files) + BATCH_SIZE - 1) // BATCH_SIZE
print("num_batches:", num_batches)

Found 164 code files to process.
num_batches: 33


In [ ]:
def extract_function_name(code_text):
    try:
        tree = ast.parse(code_text)
        function_names = [node.name for node in tree.body if isinstance(node, ast.FunctionDef)]
        return function_names[-1] if function_names else "unknown_function"
    except Exception:
        return "unknown_function"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

def warmup_cpu(duration_secs):
    print(f"\nCPU warm-up for {duration_secs} seconds...")
    start = time.time()
    a, b = 0, 1
    while (time.time() - start) < duration_secs:
        a, b = b, a + b
        if a > 10**7:
            a, b = 0, 1
    print("CPU warm-up finished.\n")

@torch.no_grad()
def warmup_model(tokenizer, model):
    print("Model/GPU warm-up (excluded): 2 tiny generations...")
    messages = [{"role": "user", "content": "Say hello in one word."}]
    inp = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
    _ = model.generate(input_ids=inp.input_ids, attention_mask=inp.attention_mask, max_new_tokens=8, do_sample=False)
    _ = model.generate(input_ids=inp.input_ids, attention_mask=inp.attention_mask, max_new_tokens=8, do_sample=False)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    print("Model/GPU warm-up finished.\n")

def backup_if_exists(path):
    if os.path.exists(path):
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        new_name = f"{path}.old_{ts}"
        os.rename(path, new_name)
        print(f"Existing file backed up: {new_name}")

In [ ]:
run_rows = []

for run_id in range(1, RUNS + 1):
    print(f"\n================ RUN {run_id}/{RUNS} ================\n")

    # ==========================================================
    # FIX #1: Per-run folders (no overlap)
    # ==========================================================
    RUN_DIR = os.path.join(MODEL_ROOT_DIR, f"run_{run_id}")
    TESTS_OUTPUT_DIR = os.path.join(RUN_DIR, "tests")
    SETUP_COST_DIR   = os.path.join(RUN_DIR, "setup_cost_csv")
    EMISSIONS_DIR    = os.path.join(RUN_DIR, "emissions_csv")
    RUNTIME_DIR      = os.path.join(RUN_DIR, "runtime_csv")
    SUMMARY_DIR      = os.path.join(RUN_DIR, "summary")

    os.makedirs(TESTS_OUTPUT_DIR, exist_ok=True)
    os.makedirs(SETUP_COST_DIR, exist_ok=True)
    os.makedirs(EMISSIONS_DIR, exist_ok=True)
    os.makedirs(RUNTIME_DIR, exist_ok=True)
    os.makedirs(SUMMARY_DIR, exist_ok=True)

    setup_cost_csv_path = os.path.join(SETUP_COST_DIR, f"setup_cost_run_{run_id}.csv")
    emissions_csv_path  = os.path.join(EMISSIONS_DIR,  f"emissions_run_{run_id}.csv")
    runtime_csv_path    = os.path.join(RUNTIME_DIR,    f"runtime_run_{run_id}.csv")

    # Ensure no overwrite/mixing if you re-run the notebook
    backup_if_exists(setup_cost_csv_path)
    backup_if_exists(emissions_csv_path)
    backup_if_exists(runtime_csv_path)

    run_start_time = time.time()

    # (Excluded) CPU warm-up
    warmup_cpu(WARMUP_SECONDS_CPU)

    # ----------------------------------------------------------
    # Setup cost tracking: model loading only
    # ----------------------------------------------------------
    setup_tracker = EmissionsTracker(
        project_name=f"{MODEL_ID.replace('/', '_')}_run_{run_id}_setup_loading",
        output_dir=os.path.dirname(setup_cost_csv_path),
        output_file=os.path.basename(setup_cost_csv_path),
    )
    setup_tracker.start()

    print(f"Loading 8-bit quantized model '{MODEL_ID}'...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto"
    )
    model.eval()

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    setup_emissions_kg = setup_tracker.stop()
    print(f"Run {run_id} setup emissions (kg CO2eq): {setup_emissions_kg}")
    print(f"Run {run_id} setup cost CSV: {setup_cost_csv_path}")

    # (Excluded) Model/GPU warm-up after load
    warmup_model(tokenizer, model)

    # runtime CSV header
    with open(runtime_csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["run_id", "batch_id", "file_path", "status", "seconds"])

        # ==========================================================
        # FIX #2: Per-batch CodeCarbon tracking (one row per batch)
        # This appends incrementally to emissions_run_{run_id}.csv
        # ==========================================================
        for batch_id in tqdm(range(num_batches), desc=f"Run {run_id} Processing batches"):
            batch_start = time.time()

            batch_tracker = EmissionsTracker(
                project_name=f"{MODEL_ID.replace('/', '_')}_run_{run_id}_batch_{batch_id}_inference",
                output_dir=os.path.dirname(emissions_csv_path),
                output_file=os.path.basename(emissions_csv_path),
            )
            batch_tracker.start()

            start_index = batch_id * BATCH_SIZE
            end_index = min(start_index + BATCH_SIZE, len(code_files))
            batch_files = code_files[start_index:end_index]

            for file_path in batch_files:
                t0 = time.time()

                if not os.path.exists(file_path):
                    w.writerow([run_id, batch_id, file_path, "missing", time.time() - t0])
                    continue

                try:
                    with open(file_path, "r", encoding="utf-8") as src:
                        code_content = src.read()

                    module_name = os.path.basename(file_path).replace(".py", "")
                    function_name = extract_function_name(code_content)

                    messages = [
                {
                    "role": "system",
                    "content": f"""### Task Context
You are an expert Python programmer. Your only task is to write complete unittest test suites.

### Tone Context
Maintain a professional, precise, and methodical tone.

### Detailed Task Description & Rules
1. Analyze the provided Python function.
2. Generate a self-contained unittest test suite.
3. The output must:
   - Begin with import unittest
   - Include from {module_name} import {function_name}
   - Define a single unittest.TestCase class
   - Include multiple test_ methods for normal, edge, and invalid inputs
   - End with if __name__ == '__main__': unittest.main()
4. Use only unittest assertions.
5. Do not include markdown, prose, or explanations.
6. Output must be runnable Python code.

### Example
#### Function:
def sum_of_elements(numbers: list) -> int:
    \"\"\"Return the sum of all integers in a list.\"\"\"
    return sum(numbers)

#### Test Script:
import unittest

class TestSumOfElements(unittest.TestCase):
    def test_positive_numbers(self):
        self.assertEqual(sum_of_elements([1, 2, 3, 4]), 10)

    def test_negative_numbers(self):
        self.assertEqual(sum_of_elements([-1, -2, -3]), -6)

    def test_empty_list(self):
        self.assertEqual(sum_of_elements([]), 0)

if __name__ == '__main__':
    unittest.main()
"""
                },
                {
                    "role": "user",
                    "content": f"""### Immediate Task
Write the complete unittest test suite for the following Python function.

### Output Formatting
1. Start with: import unittest
2. Include: from {module_name} import {function_name}
3. End with:
if __name__ == '__main__':
    unittest.main()

Function:
{code_content}
"""
                    }]

                    model_inputs = tokenizer.apply_chat_template(
                        messages,
                        add_generation_prompt=True,
                        return_tensors="pt"
                    ).to(model.device)

                    generated_ids = model.generate(
                        input_ids=model_inputs.input_ids,
                        attention_mask=model_inputs.attention_mask,
                        max_new_tokens=1024,
                        do_sample=False,
                        temperature=0.0
                    )

                    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
                    generated_test = generated_text.strip().replace("```python", "").replace("```", "").strip()

                    task_id = os.path.basename(file_path).replace("_code.py", "")
                    output_filename = f"test_{task_id}_test.py"
                    output_path = os.path.join(TESTS_OUTPUT_DIR, output_filename)

                    with open(output_path, "w", encoding="utf-8") as out:
                        out.write(generated_test)

                    w.writerow([run_id, batch_id, file_path, "ok", time.time() - t0])

                except Exception as e:
                    w.writerow([run_id, batch_id, file_path, f"error: {type(e).__name__}", time.time() - t0])

            # finish batch tracker (this writes ONE row into emissions_run_{run_id}.csv)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            _ = batch_tracker.stop()

            w.writerow([run_id, batch_id, f"[BATCH_TOTAL_{batch_id}]", "batch_done", time.time() - batch_start])
            f.flush()  # ensure runtime rows are saved even if session dies

    run_duration_s = time.time() - run_start_time

    run_rows.append({
        "run_id": run_id,
        "run_dir": RUN_DIR,
        "setup_cost_csv": setup_cost_csv_path,
        "emissions_csv": emissions_csv_path,
        "runtime_csv": runtime_csv_path,
        "run_duration_seconds": run_duration_s
    })

    print(f"\nRun {run_id} completed. Duration (s): {run_duration_s}")
    print(f"Run {run_id} emissions CSV (per-batch rows): {emissions_csv_path}")
    print(f"Run {run_id} runtime CSV: {runtime_csv_path}")

    if run_id < RUNS:
        print(f"\nCooling for {COOLING_SECONDS} seconds...\n")
        try:
            del model
            del tokenizer
        except Exception:
            pass
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        time.sleep(COOLING_SECONDS)

print("\nAll runs complete!")


================ RUN 1/1 ================


CPU warm-up for 300 seconds...


[codecarbon WARNING @ 01:08:55] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 01:08:55] [setup] RAM Tracking...
[codecarbon INFO @ 01:08:55] [setup] CPU Tracking...


CPU warm-up finished.



[codecarbon WARNING @ 01:08:56] We saw that you have a Intel(R) Xeon(R) CPU @ 2.20GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 01:08:56] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 01:08:56] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.20GHz
[codecarbon WARNING @ 01:08:56] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 01:08:56] [setup] GPU Tracking...
[codecarbon INFO @ 01:08:56] Tracking Nvidia GPU via pynvml
[codecarbon INFO @ 01:08:56] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Method: global constant
                GPU Tracking Method: pynvml
            
[codecarbon INFO @ 01:08:56] >>> Tracker's metadata:
[codecarbon INFO @ 01:08:56]   Platform sys

Loading 8-bit quantized model 'meta-llama/Meta-Llama-3-8B-Instruct'...


...

Run 1 setup emissions (kg CO2eq): 0.0006825021379469675
Run 1 setup cost CSV: /content/drive/MyDrive/Llama/APV3/run_1/setup_cost_csv/setup_cost_run_1.csv
Model/GPU warm-up (excluded): 2 tiny generations...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Model/GPU warm-up finished.



Run 1 Processing batches:   0%|          | 0/33 [00:00<?, ?it/s][codecarbon WARNING @ 01:13:20] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 01:13:20] [setup] RAM Tracking...
[codecarbon INFO @ 01:13:20] [setup] CPU Tracking...
[codecarbon WARNING @ 01:13:21] We saw that you have a Intel(R) Xeon(R) CPU @ 2.20GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 01:13:21] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 01:13:21] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.20GHz
[codecarbon WARNING @ 01:13:21] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 01:13:21] [setup] GPU Tracking...
[codecarbon INFO @ 01:13:21] Tracking Nvidia GPU via pynvml
[codecarbon INFO @ 01:13:21] The below tracking methods have been set up


Run 1 completed. Duration (s): 7899.128041267395
Run 1 emissions CSV (per-batch rows): /content/drive/MyDrive/Llama/APV3/run_1/emissions_csv/emissions_run_1.csv
Run 1 runtime CSV: /content/drive/MyDrive/Llama/APV3/run_1/runtime_csv/runtime_run_1.csv

All runs complete!
